In [ ]:
# ============================================================
# DIRECT VS CONJUGATE FORM:
# PERMISSIBLE POLE LOCATIONS AFTER COEFFICIENT QUANTIZATION
# ============================================================
#
# This notebook demonstrates how the realization structure of a
# second-order IIR section affects the set of pole locations that
# can be obtained when the filter coefficients are represented
# with finite precision.
#
# The theoretical pole pair is
#
#                   p1,2 = r exp(±j theta)
#
# and two different second-order realizations are considered.
#
#
# ============================================================
# DIRECT FORM
# ============================================================
#
# For the direct form,
#
#                   alpha1 = 2 r cos(theta)
#
#                   alpha2 = -r^2.
#
# Therefore,
#
#                   Re{p} = r cos(theta) = alpha1 / 2
#
# and
#
#                   r = sqrt(-alpha2).
#
# Hence,
#
#                   Im{p}
#
#                   = sqrt(r^2 - Re{p}^2)
#
#                   = sqrt(-alpha2 - alpha1^2 / 4).
#
# When alpha1 and alpha2 can take only quantized values, the
# resulting pole coordinates are not uniformly distributed in
# the complex plane.
#
# In particular, the nonlinear relationship between the
# coefficients and the pole coordinates produces relatively
# sparse permissible positions in some regions of the plane.
#
#
# ============================================================
# CONJUGATE FORM
# ============================================================
#
# For the conjugate form,
#
#                   alpha1 = r cos(theta)
#
#                   alpha2 = r sin(theta).
#
# Consequently,
#
#                   Re{p} = alpha1
#
#                   Im{p} = alpha2.
#
# Thus, coefficient quantization directly quantizes the
# horizontal and vertical coordinates of the pole.
#
# The permissible pole locations therefore form a regular
# Cartesian grid inside the unit circle.
#
#
# ============================================================
# WHY THE TWO FORMS ARE DIFFERENT
# ============================================================
#
# With infinite coefficient precision, both realizations can
# produce arbitrary pole positions inside the unit circle.
#
# Finite coefficient precision changes this situation because
# every coefficient can take only a discrete set of values.
#
# In the direct form, the coefficient-to-pole mapping is
# nonlinear.
#
# In the conjugate form, the mapping is linear:
#
#                   alpha1 -> Re{p}
#
#                   alpha2 -> Im{p}.
#
# Therefore:
#
#       DIRECT FORM
#           -> nonuniform permissible pole locations
#
#       CONJUGATE FORM
#           -> regular Cartesian pole grid
#
#
# ============================================================
# EFFECT OF WORD LENGTH
# ============================================================
#
# Let ell denote the coefficient word length.
#
# The normalized coefficient grid used in this demonstration has
#
#                   2^ell - 1
#
# positive nonzero levels along each coordinate direction.
#
# Increasing ell therefore increases the density of permissible
# pole locations.
#
#       small ell
#           -> coarse set of permissible positions
#
#       large ell
#           -> dense set of permissible positions
#
# As ell tends to infinity, the discrete permissible positions
# approach the continuous region inside the unit circle.
#
#
# ============================================================
# WHAT TO OBSERVE
# ============================================================
#
# DIRECT FORM
#
# Red crosses show the permissible pole positions.
#
# Observe that the points do not form a uniform rectangular grid.
#
#
# CONJUGATE FORM
#
# Blue open circles show the permissible pole positions.
#
# Observe the regular horizontal and vertical spacing.
#
#
# BOTH FORMS
#
# The two sets are superimposed.
#
# The different symbols allow their geometrical distributions to
# be compared directly:
#
#       red x            -> Direct form
#
#       blue open circle -> Conjugate form
#
#
# ============================================================
# IMPORTANT INTERPRETATION
# ============================================================
#
# The poles themselves are not directly quantized.
#
# What is quantized are the realization coefficients.
#
# The realization structure determines how those discrete
# coefficient values are mapped into permissible pole positions
# in the complex plane.
#
# ============================================================


%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interactive, IntSlider, RadioButtons, VBox, HBox, HTML, Layout
from IPython.display import display


# ------------------------------------------------------------
# Generate permissible conjugate-form pole positions
# ------------------------------------------------------------

def conjugate_pole_locations(bits):

    number_of_levels = 2**bits - 1

    q = 1.0 / number_of_levels

    levels = np.arange(1, number_of_levels + 1) * q

    x_values = []

    y_values = []

    for alpha1 in levels:

        for alpha2 in levels:

            x = alpha1

            y = alpha2

            r = np.sqrt(x**2 + y**2)

            if r < 1.0:

                x_values.append(x)

                y_values.append(y)

    return np.array(x_values), np.array(y_values), q


# ------------------------------------------------------------
# Generate permissible direct-form pole positions
# ------------------------------------------------------------

def direct_pole_locations(bits):

    number_of_levels = 2**bits - 1

    q = 1.0 / number_of_levels

    alpha1_levels = 2.0 * np.arange(1, number_of_levels + 1) * q

    alpha2_levels = -np.arange(1, number_of_levels + 1) * q

    x_values = []

    y_values = []

    tolerance = 1e-12

    for alpha1 in alpha1_levels:

        for alpha2 in alpha2_levels:

            x = alpha1 / 2.0

            r_squared = -alpha2

            y_squared = r_squared - x**2

            if y_squared >= -tolerance:

                y_squared = max(0.0, y_squared)

                y = np.sqrt(y_squared)

                r = np.sqrt(x**2 + y**2)

                if r < 1.0:

                    x_values.append(x)

                    y_values.append(y)

    return np.array(x_values), np.array(y_values), q


# ------------------------------------------------------------
# Style
# ------------------------------------------------------------

style_html = HTML("""
<style>

.pl-root {
    font-family: monospace;
    width: 960px;
    max-width: 960px;
}

.pl-description {
    font-size: 13px;
    line-height: 1.45;
    padding: 9px 12px;
    border: 1px solid #bfc7d5;
    border-left: 6px solid #4a6fa5;
    background: #f7f9fc;
    border-radius: 8px;
    margin-bottom: 8px;
    box-sizing: border-box;
}

.pl-box {
    border: 1px solid #c8d0dc;
    border-radius: 9px;
    padding: 9px 12px;
    box-sizing: border-box;
}

.pl-title {
    font-size: 16px;
    font-weight: bold;
    color: #243447;
    margin-bottom: 6px;
}

.pl-info {
    font-size: 13px;
    line-height: 1.50;
}

.pl-label {
    display: inline-block;
    min-width: 225px;
    font-weight: bold;
}

.pl-value {
    font-size: 14px;
    font-weight: bold;
    color: #1b3a57;
}

.pl-note {
    font-size: 12px;
    line-height: 1.35;
    color: #555555;
    margin-top: 5px;
}

.pl-direct {
    color: #b00020;
    font-weight: bold;
}

.pl-conjugate {
    color: #1760a8;
    font-weight: bold;
}

.jupyter-widgets-output-area,
.widget-output,
.output_area,
.output_subarea {
    overflow-x: visible !important;
    max-width: none !important;
}

</style>
""")


# ------------------------------------------------------------
# Title
# ------------------------------------------------------------

title_html = HTML("""
<div class="pl-root">

    <div style="
        font-family:monospace;
        font-size:22px;
        font-weight:bold;
        margin-bottom:8px;
    ">
        Direct vs Conjugate Form: Permissible Pole Locations
    </div>

</div>
""")


# ------------------------------------------------------------
# Description
# ------------------------------------------------------------

description_html = HTML("""
<div class="pl-root">

    <div class="pl-description">

        Finite coefficient precision restricts the positions that a pole can
        occupy in the complex plane.<br>

        In the <b>direct form</b>, the nonlinear dependence between the
        coefficients and the pole coordinates produces a nonuniform
        distribution of permissible positions.<br>

        In the <b>conjugate form</b>, the coefficients correspond directly
        to the horizontal and vertical pole coordinates and therefore
        generate a regular Cartesian grid.

    </div>

</div>
""")


# ------------------------------------------------------------
# Dynamic summary
# ------------------------------------------------------------

summary_html = HTML()

summary_html.layout = Layout(
    width='610px',
    min_width='610px',
    overflow='visible'
)


# ------------------------------------------------------------
# Main plotting function
# ------------------------------------------------------------

def plot_permissible_poles(bits=4, display_mode='Both forms'):

    direct_x, direct_y, q_direct = direct_pole_locations(bits)

    conjugate_x, conjugate_y, q_conjugate = conjugate_pole_locations(bits)

    number_of_levels = 2**bits - 1


    # --------------------------------------------------------
    # Summary interpretation
    # --------------------------------------------------------

    if display_mode == 'Direct form':

        interpretation = """
        <span class="pl-direct">DIRECT FORM:</span>
        red crosses show the nonuniform pole-position distribution
        produced by the nonlinear coefficient-to-pole mapping.
        """

    elif display_mode == 'Conjugate form':

        interpretation = """
        <span class="pl-conjugate">CONJUGATE FORM:</span>
        blue open circles show the regular Cartesian grid produced
        because the two coefficients directly determine the real
        and imaginary pole coordinates.
        """

    else:

        interpretation = """
        <span class="pl-direct">Red crosses:</span> direct form.
        &nbsp;&nbsp;
        <span class="pl-conjugate">Blue open circles:</span> conjugate form.
        The two different geometries result from the different
        coefficient-to-pole mappings.
        """


    summary_html.value = f"""
    <div class="pl-box">

        <div class="pl-title">
            Finite-Precision Pole Grid
        </div>

        <div class="pl-info">

            <span class="pl-label">Coefficient word length</span>
            ℓ = <span class="pl-value">{bits} bits</span>
            <br>

            <span class="pl-label">Nonzero levels / coefficient</span>
            2<sup>{bits}</sup> - 1 = {number_of_levels}
            <br>

            <span class="pl-label">Normalized grid step</span>
            q = 1 / {number_of_levels} = {q_direct:.6f}
            <br>

            <span class="pl-label">Direct-form positions</span>
            {len(direct_x)}
            <br>

            <span class="pl-label">Conjugate-form positions</span>
            {len(conjugate_x)}
            <br>

            <span class="pl-label">Displayed form</span>
            {display_mode}

        </div>

        <div class="pl-note">
            {interpretation}
        </div>

    </div>
    """


    # --------------------------------------------------------
    # Create a larger square pole-location diagram
    # --------------------------------------------------------
    #
    # The previous side length was approximately 4.5 inches.
    # The new diagram is approximately 50% larger.
    #
    # --------------------------------------------------------

    fig, ax = plt.subplots(
        figsize=(10.0, 6.75)
    )


    # --------------------------------------------------------
    # Unit-circle boundary in the first quadrant
    # --------------------------------------------------------

    theta = np.linspace(
        0.0,
        np.pi / 2.0,
        500
    )


    ax.plot(
        np.cos(theta),
        np.sin(theta),
        'k--',
        linewidth=1.4,
        label='Unit circle'
    )


    # --------------------------------------------------------
    # Conjugate-form positions
    #
    # Draw these FIRST when both forms are selected.
    # Blue open circles make the regular Cartesian grid visible.
    # --------------------------------------------------------

    if display_mode == 'Conjugate form' or display_mode == 'Both forms':

        ax.scatter(
            conjugate_x,
            conjugate_y,
            s=58,
            marker='o',
            facecolors='none',
            edgecolors='blue',
            linewidths=1.4,
            label='Conjugate form',
            zorder=3
        )


    # --------------------------------------------------------
    # Direct-form positions
    #
    # Draw these AFTER the conjugate grid so that overlapping
    # red crosses remain visible.
    # --------------------------------------------------------

    if display_mode == 'Direct form' or display_mode == 'Both forms':

        ax.scatter(
            direct_x,
            direct_y,
            s=48,
            marker='x',
            color='red',
            linewidths=1.4,
            label='Direct form',
            zorder=4
        )


    # --------------------------------------------------------
    # Axes
    # --------------------------------------------------------

    ax.axhline(
        0.0,
        color='gray',
        linewidth=0.8
    )


    ax.axvline(
        0.0,
        color='gray',
        linewidth=0.8
    )


    ax.set_xlim(
        -0.02,
        1.05
    )


    ax.set_ylim(
        -0.02,
        1.05
    )


    ax.set_aspect(
        'equal',
        adjustable='box'
    )


    ax.set_xlabel(
        'Real part',
        fontsize=11
    )


    ax.set_ylabel(
        'Imaginary part',
        fontsize=11
    )


    ax.set_title(
        'Permissible Pole Locations in the First Quadrant',
        fontsize=12
    )


    ax.grid(
        True,
        linestyle=':',
        alpha=0.45
    )


    # --------------------------------------------------------
    # Legend below the horizontal axis
    # --------------------------------------------------------

    if display_mode == 'Both forms':

        legend_columns = 3

    else:

        legend_columns = 2


    ax.legend(
        loc='upper center',
        bbox_to_anchor=(0.5, -0.12),
        ncol=legend_columns,
        fontsize=9,
        frameon=False
    )


    # --------------------------------------------------------
    # Final figure spacing
    # --------------------------------------------------------

    plt.subplots_adjust(
        left=0.08,
        right=0.98,
        top=0.92,
        bottom=0.20
    )


    plt.show()

    plt.close(fig)


# ------------------------------------------------------------
# Controls
# ------------------------------------------------------------

slider_layout = Layout(
    width='310px'
)


slider_style = {
    'description_width': '125px'
}


bits_slider = IntSlider(
    value=4,
    min=2,
    max=8,
    step=1,
    description='Coefficient bits ℓ:',
    continuous_update=True,
    style=slider_style,
    layout=slider_layout
)


display_selector = RadioButtons(
    options=[
        'Both forms',
        'Direct form',
        'Conjugate form'
    ],
    value='Both forms',
    description='Display:',
    style={'description_width': '70px'},
    layout=Layout(
        width='310px'
    )
)


# ------------------------------------------------------------
# Interactive object
# ------------------------------------------------------------

widget_plot = interactive(
    plot_permissible_poles,
    bits=bits_slider,
    display_mode=display_selector
)


# ------------------------------------------------------------
# Controls box
# ------------------------------------------------------------

controls = VBox(
    [
        HTML("<div class='pl-title'>Controls</div>"),
        bits_slider,
        HTML("<div style='height:5px;'></div>"),
        display_selector
    ],
    layout=Layout(
        width='335px',
        min_width='335px',
        border='1px solid #c8d0dc',
        padding='9px',
        overflow='visible'
    )
)


# ------------------------------------------------------------
# Summary + controls
# ------------------------------------------------------------

top_row = HBox(
    [
        summary_html,
        controls
    ],
    layout=Layout(
        width='960px',
        max_width='960px',
        overflow='visible',
        align_items='flex-start',
        justify_content='space-between'
    )
)


# ------------------------------------------------------------
# Plot output
# ------------------------------------------------------------

plot_output = widget_plot.children[-1]

plot_output.layout = Layout(
    width='auto',
    overflow='visible'
)


# ------------------------------------------------------------
# Final notebook layout
# ------------------------------------------------------------

main_layout = VBox(
    [
        description_html,
        top_row,
        plot_output
    ],
    layout=Layout(
        width='960px',
        overflow='visible',
        align_items='flex-start'
    )
)


# ------------------------------------------------------------
# Display
# ------------------------------------------------------------

display(style_html)

display(title_html)

display(main_layout)